# Proyecto Big Data – Análisis de Ventas 2021-2026

**Objetivo del Proyecto:** Construir un pipeline de procesamiento **batch** sobre datos históricos de ventas (facturas electrónicas) para obtener métricas clave: ventas por mes, productos más vendidos, comparación por año, y más.

**Datos fuente:** `Ventas_2021-2026.xlsx` — Comprobantes electrónicos emitidos entre 2021 y 2026.

**Stack tecnológico:**
- Apache Spark (PySpark) — procesamiento distribuido batch
- pandas — lectura inicial del Excel
- Jupyter Notebook — entorno interactivo
- Docker — infraestructura contenerizada

**Arquitectura:** Lambda — capa batch sobre datos históricos.

## ¿Por qué arquitectura Lambda para este proyecto?

| Necesidad | ¿La tiene este proyecto? | Implicancia |
|---|---|---|
| Datos históricos acumulados (2021-2026) | ✅ Sí | **Batch layer** |
| Actualizaciones en tiempo real | ❌ No (por ahora) | Speed layer opcional |
| Consultas rápidas de reportes | ✅ Sí | **Serving layer** |

**Conclusión:** Iniciamos con la **capa batch** de Lambda. Procesamos todo el histórico para generar métricas consolidadas.

```
Ventas_2021-2026.xlsx
        │
        ▼
  [Ingesta - pandas]
        │
        ▼
  [Limpieza - PySpark]
        │
        ▼
  [Transformación - PySpark]
        │
        ▼
  [Aggregaciones batch]
        │
        ▼
  [Resultados / Métricas]
```

## Paso 1 — Configurar el entorno

In [9]:
!pip install openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 4.7 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [10]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType, DateType
)

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


In [11]:
# Inicializar SparkSession
spark = (
    SparkSession.builder
    .appName("ProyectoVentas_Batch")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"✅ Spark iniciado correctamente")
print(f"   Versión: {spark.version}")
print(f"   App Name: {spark.sparkContext.appName}")

✅ Spark iniciado correctamente
   Versión: 4.1.1
   App Name: ProyectoVentas_Batch


## Paso 2 — Cargar el dataset (Excel → pandas → Spark)

In [12]:
# Ruta al archivo (ajustar según montaje Docker)
EXCEL_PATH = "../data/Ventas_2021-2026.xlsx"

# Leer con pandas (soporte nativo para .xlsx)
print("⏳ Cargando Excel... esto puede tomar unos segundos...")
df_pd = pd.read_excel(EXCEL_PATH, dtype=str)

print(f"✅ Excel cargado con éxito")
print(f"   Filas: {df_pd.shape[0]:,}")
print(f"   Columnas: {df_pd.shape[1]}")

⏳ Cargando Excel... esto puede tomar unos segundos...
✅ Excel cargado con éxito
   Filas: 525,434
   Columnas: 24


In [13]:
# Ver las columnas exactas del dataset
print("📋 Columnas del dataset:")
for i, col in enumerate(df_pd.columns, 1):
    print(f"  {i:>2}. {col}")

📋 Columnas del dataset:
   1. FECHA DE EMISIÓN
   2. TIPO
   3. SERIE
   4. NÚMERO
   5. DOC ENTIDAD TIPO
   6. DOC ENTIDAD NÚMERO
   7. DENOMINACIÓN ENTIDAD
   8. MONEDA
   9. TIPO DE CAMBIO
  10. UNIDAD DE MEDIDA
  11. CÓDIGO
  12. DESCRIPCIÓN
  13. CANTIDAD
  14. VALOR UNITARIO
  15. PRECIO UNITARIO
  16. DESCUENTO
  17. SUBTOTAL
  18. TIPO DE IGV
  19. IGV
  20. TIPO DE ISC
  21. ISC
  22. IMPUESTO BOLSAS
  23. TOTAL
  24. ANULADO


In [14]:
# Vista previa
df_pd.head(3)

,FECHA DE EMISIÓN,TIPO,SERIE,NÚMERO,DOC ENTIDAD TIPO,DOC ENTIDAD NÚMERO,DENOMINACIÓN ENTIDAD,MONEDA,TIPO DE CAMBIO,UNIDAD DE MEDIDA,...,PRECIO UNITARIO,DESCUENTO,SUBTOTAL,TIPO DE IGV,IGV,TIPO DE ISC,ISC,IMPUESTO BOLSAS,TOTAL,ANULADO
0,20/01/2021,03,BQQ1,1,-,0001,juan,PEN,0,NIU,...,0,0,0,10,0,NaN,0,0,0,SI
1,24/01/2021,03,BQQ1,2,-,10698523,CLAUDIA F JR GIRASOLES,PEN,0,NIU,...,0,0,0,10,0,NaN,0,0,0,SI
2,25/01/2021,03,BQQ1,3,-,62,MARGARITA PINEDA QUISPE,PEN,0,NIU,...,0.86667,0,11.0169915254,10,1.9830584746,NaN,0,0,13.00005,NO


## Paso 3 — Estandarizar nombres de columnas

In [15]:
# Normalizar nombres: minúsculas, sin espacios, sin tildes
def normalizar_columna(nombre):
    import unicodedata
    nombre = str(nombre).strip().lower()
    nombre = unicodedata.normalize('NFD', nombre)
    nombre = ''.join(c for c in nombre if unicodedata.category(c) != 'Mn')
    nombre = nombre.replace(' ', '_').replace('/', '_').replace('-', '_')
    return nombre

columnas_originales = df_pd.columns.tolist()
columnas_normalizadas = [normalizar_columna(c) for c in columnas_originales]

df_pd.columns = columnas_normalizadas

print("📋 Mapeo de columnas:")
for orig, norm in zip(columnas_originales, columnas_normalizadas):
    print(f"  '{orig}' → '{norm}'")

📋 Mapeo de columnas:
  'FECHA DE EMISIÓN' → 'fecha_de_emision'
  'TIPO' → 'tipo'
  'SERIE' → 'serie'
  'NÚMERO' → 'numero'
  'DOC ENTIDAD TIPO' → 'doc_entidad_tipo'
  'DOC ENTIDAD NÚMERO' → 'doc_entidad_numero'
  'DENOMINACIÓN ENTIDAD' → 'denominacion_entidad'
  'MONEDA' → 'moneda'
  'TIPO DE CAMBIO' → 'tipo_de_cambio'
  'UNIDAD DE MEDIDA' → 'unidad_de_medida'
  'CÓDIGO' → 'codigo'
  'DESCRIPCIÓN' → 'descripcion'
  'CANTIDAD' → 'cantidad'
  'VALOR UNITARIO' → 'valor_unitario'
  'PRECIO UNITARIO' → 'precio_unitario'
  'DESCUENTO' → 'descuento'
  'SUBTOTAL' → 'subtotal'
  'TIPO DE IGV' → 'tipo_de_igv'
  'IGV' → 'igv'
  'TIPO DE ISC' → 'tipo_de_isc'
  'ISC' → 'isc'
  'IMPUESTO BOLSAS' → 'impuesto_bolsas'
  'TOTAL' → 'total'
  'ANULADO' → 'anulado'


## Paso 4 — Convertir a Spark DataFrame

In [16]:
# Pasar de pandas a Spark
df_spark = spark.createDataFrame(df_pd)

print(f"✅ DataFrame Spark creado")
print(f"   Particiones: {df_spark.rdd.getNumPartitions()}")
print(f"   Total registros: {df_spark.count():,}")

✅ DataFrame Spark creado
   Particiones: 28


26/04/14 12:44:15 WARN TaskSetManager: Stage 0 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
[Stage 0:>                                                        (0 + 28) / 28]

   Total registros: 525,434


In [17]:
# Schema inicial (todo string, lo tipamos en el siguiente paso)
df_spark.printSchema()

root
 |-- fecha_de_emision: string (nullable = true)
 |-- tipo: string (nullable = true)
 |-- serie: string (nullable = true)
 |-- numero: string (nullable = true)
 |-- doc_entidad_tipo: string (nullable = true)
 |-- doc_entidad_numero: string (nullable = true)
 |-- denominacion_entidad: string (nullable = true)
 |-- moneda: string (nullable = true)
 |-- tipo_de_cambio: string (nullable = true)
 |-- unidad_de_medida: string (nullable = true)
 |-- codigo: string (nullable = true)
 |-- descripcion: string (nullable = true)
 |-- cantidad: string (nullable = true)
 |-- valor_unitario: string (nullable = true)
 |-- precio_unitario: string (nullable = true)
 |-- descuento: string (nullable = true)
 |-- subtotal: string (nullable = true)
 |-- tipo_de_igv: string (nullable = true)
 |-- igv: string (nullable = true)
 |-- tipo_de_isc: double (nullable = true)
 |-- isc: string (nullable = true)
 |-- impuesto_bolsas: string (nullable = true)
 |-- total: string (nullable = true)
 |-- anulado: strin

## Paso 5 — Exploración inicial (EDA básico)

In [18]:
# Detectar columna de fecha automáticamente
col_fecha = [c for c in df_spark.columns if 'fecha' in c]
col_total = [c for c in df_spark.columns if 'total' in c]
col_desc  = [c for c in df_spark.columns if 'descripcion' in c or 'denominacion' in c]
col_anulado = [c for c in df_spark.columns if 'anulado' in c]

print("🔍 Columnas detectadas automáticamente:")
print(f"  Fecha    : {col_fecha}")
print(f"  Total    : {col_total}")
print(f"  Descripción: {col_desc}")
print(f"  Anulado  : {col_anulado}")

🔍 Columnas detectadas automáticamente:
  Fecha    : ['fecha_de_emision']
  Total    : ['subtotal', 'total']
  Descripción: ['denominacion_entidad', 'descripcion']
  Anulado  : ['anulado']


In [19]:
# Verificar nulos por columna
print("📊 Valores nulos por columna:")
nulos = df_spark.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_spark.columns
])
nulos.show(vertical=True)

📊 Valores nulos por columna:


26/04/14 12:44:44 WARN TaskSetManager: Stage 3 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.


-RECORD 0-------------------
 fecha_de_emision     | 0   
 tipo                 | 0   
 serie                | 0   
 numero               | 0   
 doc_entidad_tipo     | 0   
 doc_entidad_numero   | 0   
 denominacion_entidad | 0   
 moneda               | 0   
 tipo_de_cambio       | 0   
 unidad_de_medida     | 0   
 codigo               | 0   
 descripcion          | 0   
 cantidad             | 0   
 valor_unitario       | 0   
 precio_unitario      | 0   
 descuento            | 0   
 subtotal             | 0   
 tipo_de_igv          | 0   
 igv                  | 0   
 tipo_de_isc          | 0   
 isc                  | 0   
 impuesto_bolsas      | 0   
 total                | 0   
 anulado              | 0   



In [20]:
# Tipos de documentos presentes
if 'tipo' in df_spark.columns:
    print("📄 Tipos de comprobantes:")
    df_spark.groupBy('tipo').count().orderBy(F.desc('count')).show()

📄 Tipos de comprobantes:


26/04/14 12:44:48 WARN TaskSetManager: Stage 6 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.


+----+------+
|tipo| count|
+----+------+
|  03|491496|
|  01| 33898|
|  07|    40|
+----+------+



## Paso 6 — Limpieza y tipado de columnas clave

In [21]:
# Ajustar estos nombres según el output del Paso 3
# ¡IMPORTANTE! Si tus columnas normalizadas son distintas, cámbialas aquí
FECHA_COL   = col_fecha[0]  if col_fecha   else None
TOTAL_COL   = col_total[0]  if col_total   else None
ANULADO_COL = col_anulado[0] if col_anulado else None

df_clean = df_spark

# 1. Convertir fecha
if FECHA_COL:
    df_clean = df_clean.withColumn(
        FECHA_COL,
        F.to_date(F.col(FECHA_COL), "dd/MM/yyyy")
    )
    print(f"✅ Columna '{FECHA_COL}' convertida a fecha")

# 2. Convertir total a numérico
if TOTAL_COL:
    df_clean = df_clean.withColumn(
        TOTAL_COL,
        F.regexp_replace(F.col(TOTAL_COL), ",", "").cast("double")
    )
    print(f"✅ Columna '{TOTAL_COL}' convertida a double")

# 3. Filtrar anulados (si existe la columna)
registros_antes = df_clean.count()
if ANULADO_COL:
    df_clean = df_clean.filter(
        (F.col(ANULADO_COL).isNull()) |
        (~F.upper(F.col(ANULADO_COL)).isin("SI", "S", "1", "TRUE", "ANULADO"))
    )
    registros_despues = df_clean.count()
    print(f"✅ Anulados filtrados: {registros_antes - registros_despues:,} registros removidos")
    print(f"   Registros válidos: {registros_despues:,}")
else:
    print(f"ℹ️  No se detectó columna de anulados")
    print(f"   Registros totales: {registros_antes:,}")

✅ Columna 'fecha_de_emision' convertida a fecha
✅ Columna 'subtotal' convertida a double


26/04/14 12:44:53 WARN TaskSetManager: Stage 9 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
26/04/14 12:44:54 WARN TaskSetManager: Stage 12 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
[Stage 12:>                                                       (0 + 28) / 28]

✅ Anulados filtrados: 2,746 registros removidos
   Registros válidos: 522,688


In [22]:
# Agregar columnas de tiempo útiles para el análisis
if FECHA_COL:
    df_clean = (
        df_clean
        .withColumn("anio",       F.year(F.col(FECHA_COL)))
        .withColumn("mes",        F.month(F.col(FECHA_COL)))
        .withColumn("anio_mes",   F.date_format(F.col(FECHA_COL), "yyyy-MM"))
        .withColumn("trimestre",  F.quarter(F.col(FECHA_COL)))
    )
    print("✅ Columnas temporales agregadas: anio, mes, anio_mes, trimestre")

df_clean.select(FECHA_COL, "anio", "mes", "anio_mes", "trimestre").show(5)

✅ Columnas temporales agregadas: anio, mes, anio_mes, trimestre
+----------------+----+---+--------+---------+
|fecha_de_emision|anio|mes|anio_mes|trimestre|
+----------------+----+---+--------+---------+
|      2021-01-25|2021|  1| 2021-01|        1|
|      2021-01-25|2021|  1| 2021-01|        1|
|      2021-01-25|2021|  1| 2021-01|        1|
|      2021-01-25|2021|  1| 2021-01|        1|
|      2021-01-25|2021|  1| 2021-01|        1|
+----------------+----+---+--------+---------+
only showing top 5 rows


26/04/14 12:45:04 WARN TaskSetManager: Stage 15 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
  File "/usr/local/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
BrokenPipeError: [Errno 32] Broken pipe


## Paso 7 — Verificación final del dataset limpio

In [23]:
# Resumen del dataset listo para análisis
print("=" * 50)
print("📦 DATASET LISTO PARA ANÁLISIS BATCH")
print("=" * 50)

total_registros = df_clean.count()
print(f"  Total registros válidos : {total_registros:,}")
print(f"  Total columnas          : {len(df_clean.columns)}")

if FECHA_COL:
    rango = df_clean.agg(
        F.min(FECHA_COL).alias("desde"),
        F.max(FECHA_COL).alias("hasta")
    ).collect()[0]
    print(f"  Rango de fechas         : {rango['desde']} → {rango['hasta']}")

if TOTAL_COL:
    totales = df_clean.agg(
        F.sum(TOTAL_COL).alias("total_ventas"),
        F.avg(TOTAL_COL).alias("promedio_venta"),
        F.max(TOTAL_COL).alias("venta_maxima")
    ).collect()[0]
    print(f"  Total ventas (S/)       : {totales['total_ventas']:,.2f}")
    print(f"  Promedio por venta      : {totales['promedio_venta']:,.2f}")
    print(f"  Venta máxima            : {totales['venta_maxima']:,.2f}")

print("=" * 50)

📦 DATASET LISTO PARA ANÁLISIS BATCH


26/04/14 12:45:08 WARN TaskSetManager: Stage 16 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.


  Total registros válidos : 522,688
  Total columnas          : 28


26/04/14 12:45:09 WARN TaskSetManager: Stage 19 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.


  Rango de fechas         : 2021-01-25 → 2026-04-06


26/04/14 12:45:09 WARN TaskSetManager: Stage 22 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.


  Total ventas (S/)       : 30,696,628.42
  Promedio por venta      : 58.73
  Venta máxima            : 2,711,864.41


In [24]:
# Guardar el DataFrame limpio como variable global para los siguientes notebooks
# También lo persistimos en memoria Spark para reutilización eficiente
df_ventas = df_clean.cache()
df_ventas.count()  # Trigger cache

print("✅ df_ventas cacheado en memoria Spark")
print("   → Listo para usar en los próximos análisis batch")

26/04/14 12:45:22 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/14 12:45:22 WARN TaskSetManager: Stage 25 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
[Stage 25:>                                                       (0 + 28) / 28]

✅ df_ventas cacheado en memoria Spark
   → Listo para usar en los próximos análisis batch


26/04/14 12:45:23 WARN TaskSetManager: Stage 26 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

## Resumen del Notebook 01

| Paso | Acción | Estado |
|---|---|---|
| 1 | Configurar entorno (Spark + pandas) | ✅ |
| 2 | Cargar Excel histórico | ✅ |
| 3 | Normalizar nombres de columnas | ✅ |
| 4 | Convertir a Spark DataFrame | ✅ |
| 5 | EDA inicial (nulos, tipos de doc) | ✅ |
| 6 | Limpieza: fechas, totales, anulados | ✅ |
| 7 | Dataset batch listo con columnas temporales | ✅ |

**Próximo Notebook (02):** Análisis batch → ventas por mes, por año, top productos.